# CareerLens AI

# Notebook 03 — Baseline Machine Learning

## Objective

Train multiple machine learning models for resume classification.

Models:

- Logistic Regression
- Multinomial Naive Bayes
- Linear Support Vector Machine
- Random Forest

Compare them using:

- Accuracy
- Precision
- Recall
- F1 Score

Select the best baseline model.

In [16]:
import joblib
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [17]:
X_train = joblib.load("../datasets/processed/X_train.pkl")
X_test = joblib.load("../datasets/processed/X_test.pkl")

y_train = joblib.load("../datasets/processed/y_train.pkl")
y_test = joblib.load("../datasets/processed/y_test.pkl")

label_encoder = joblib.load("../models/label_encoder.pkl")

In [18]:
print(X_train.shape)
print(X_test.shape)

(1987, 5000)
(497, 5000)


In [19]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),

    "Naive Bayes": MultinomialNB(),

    "Linear SVM": LinearSVC(),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )
}

In [20]:
results = {}

In [21]:
for name, model in models.items():

    print("="*60)
    print(name)

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    results[name] = accuracy

    print(f"Accuracy : {accuracy:.4f}")

Logistic Regression
Accuracy : 0.6680
Naive Bayes
Accuracy : 0.5674
Linear SVM
Accuracy : 0.7425
Random Forest
Accuracy : 0.7706


In [22]:
results_df = pd.DataFrame(
    results.items(),
    columns=["Model","Accuracy"]
)

results_df.sort_values(
    by="Accuracy",
    ascending=False
)

,Model,Accuracy
3,Random Forest,0.770624
2,Linear SVM,0.742455
0,Logistic Regression,0.668008
1,Naive Bayes,0.567404


In [35]:
results = []
trained_models = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    trained_models[name] = model

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, predictions, average="weighted", zero_division=0),
        "F1 Score": f1_score(y_test, predictions, average="weighted", zero_division=0) 
        })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
)

results_df

,Model,Accuracy,Precision,Recall,F1 Score
3,Random Forest,0.770624,0.781925,0.770624,0.752989
2,Linear SVM,0.742455,0.741908,0.742455,0.735284
0,Logistic Regression,0.668008,0.675904,0.668008,0.651923
1,Naive Bayes,0.567404,0.630343,0.567404,0.532543


In [36]:
best_model_name = results_df.iloc[0]["Model"]

print(best_model_name)

Random Forest


In [37]:
best_model = trained_models[best_model_name]

joblib.dump(
    best_model,
    "../models/best_model.pkl"
)

print("Best model saved.")

Best model saved.


In [38]:
predictions = best_model.predict(X_test)

print(
    classification_report(
        y_test,
        predictions,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

                        precision    recall  f1-score   support

            ACCOUNTANT       0.72      0.96      0.82        24
              ADVOCATE       0.82      0.75      0.78        24
           AGRICULTURE       1.00      0.46      0.63        13
               APPAREL       1.00      0.37      0.54        19
                  ARTS       0.67      0.29      0.40        21
            AUTOMOBILE       1.00      0.14      0.25         7
              AVIATION       0.80      0.83      0.82        24
               BANKING       0.82      0.61      0.70        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.83      1.00      0.91        24
                  CHEF       0.86      0.79      0.83        24
          CONSTRUCTION       0.91      0.95      0.93        22
            CONSULTANT       0.78      0.61      0.68        23
              DESIGNER       0.84      1.00      0.91        21
         DIGITAL-MEDIA       0.81      

In [39]:
cm = confusion_matrix(
    y_test,
    predictions
)

cm.shape

(24, 24)

In [40]:
results_df.to_csv(
    "../reports/model_results.csv",
    index=False
)